# Project Main: Simple Data Serving Dashboard

In Step 03, we stored clean data in DuckDB.

Now we serve data to a web page.

The simple goal:

- Show a Singapore map.
- Show current taxi counts by area.
- Show current rainfall by area.
- Refresh every 2 minutes.

This introduces an important data engineering idea:

```text
One API call is not enough for live data.
```

Live dashboards need repeated refreshes.

## 1. What Is Data Serving?

Data serving means making prepared data available to users or applications.

Examples:

- A dashboard reads from a database.
- A mobile app calls an API endpoint.
- A model service asks for features.
- A report query reads the latest table.

In this lesson, the user is a web page.

## 2. What Does a Simple Web App Need?

| Part | Role |
|---|---|
| HTML | Page structure |
| CSS | Page layout and styling |
| JavaScript | Calls the backend and updates the page |
| FastAPI | Python web API server |
| DuckDB | Stores and queries the latest data |
| External APIs | Source of fresh taxi and rainfall data |

The browser does not directly clean data or write to DuckDB.

The browser asks the backend to do it.

## 3. Basic Web Programming Pieces

A very simple web application usually has two sides.

| Side | Runs Where | In Our Project | Job |
|---|---|---|---|
| Frontend | Browser | `simple_live_dashboard.html` | Show the page, handle button clicks, refresh the image |
| Backend | Python process | `simple_live_serving_app.py` | Receive requests, run data code, return HTML/JSON/PNG |

Inside the frontend file, there are three basic web languages:

| Language | What It Does | Example in This Project |
|---|---|---|
| HTML | Page structure | `<header>`, `<main>`, `<img id="map-image">`, `<button>` |
| CSS | Visual layout and style | `grid-template-columns`, colors, spacing, fonts |
| JavaScript | Browser behavior | `fetch('/api/refresh')`, `setInterval(...)`, update text and image |

A useful mental model:

```text
HTML = what exists on the page
CSS = how it looks
JavaScript = what happens after the page loads
Python backend = data work and API responses
```

## 4. Frontend: The HTML Page

The frontend file is:

```text
simple_live_dashboard.html
```

Important parts:

| Code Piece | Purpose |
|---|---|
| `<img id="map-image">` | Shows the PNG map drawn by the backend |
| `<div id="total-taxis">` | Shows the latest taxi count |
| `<div id="raining-areas">` | Shows how many areas have rain |
| `<button id="refresh-button">` | Lets the user manually refresh |
| `refreshData()` | JavaScript function that calls the backend |
| `setInterval(refreshData, 120000)` | Runs refresh every 2 minutes |

The page itself does not know how to call data.gov.sg or clean geospatial data.

It only knows how to call our backend API.

## 5. Backend: The FastAPI App

The backend file is:

```text
simple_live_serving_app.py
```

FastAPI turns Python functions into web endpoints.

Example shape:

```python
@app.get('/api/refresh')
def refresh():
    ...
```

This means:

```text
When the browser requests /api/refresh, run the refresh() Python function.
```

The backend can do things the browser should not do directly:

- Call external APIs.
- Read the GeoJSON map file.
- Use GeoPandas to convert coordinates into areas.
- Write to DuckDB.
- Draw a Matplotlib PNG.
- Return JSON responses.

## 6. Request Flow

When the page loads, and then every 2 minutes, this happens:

```text
simple_live_dashboard.html
  -> JavaScript refreshData()
  -> fetch('/api/refresh')

simple_live_serving_app.py
  -> refresh()
  -> collect_once()
  -> fetch_json(TAXI_API_URL)
  -> fetch_json(RAINFALL_API_URL)
  -> extract_taxi_counts(...)
  -> extract_rainfall_by_area(...)
  -> save_latest_to_database(...)
  -> draw_latest_map(...)
  -> map_payload()

simple_live_dashboard.html
  -> renderSummary(...)
  -> set <img id="map-image"> to /map.png?... 
```

This turns our pipeline into a served application.

One browser click or automatic timer event can trigger a full data pipeline.

## 7. Files for This Simple Version

| File | Purpose |
|---|---|
| `simple_live_dashboard.html` | Frontend: HTML, CSS, and JavaScript for the browser page |
| `simple_live_serving_app.py` | Backend: FastAPI routes, API download, cleaning, DuckDB insert, PNG drawing |
| `simple_live_latest_map.png` | Latest image drawn by the backend |
| `shared_data/simple_live_serving.duckdb` | DuckDB database created by the backend |

This version is intentionally small. It focuses on one loop: refresh data and show it.

## 8. API Endpoints

The backend has three endpoints:

| Endpoint | Meaning |
|---|---|
| `/` | Serve the web page |
| `/api/refresh` | Fetch APIs, clean data, write DuckDB, redraw PNG map, return summary |
| `/api/current` | Return latest summary and map image URL already in DuckDB |
| `/map.png` | Return the latest PNG map image |

The dashboard uses `/api/refresh` because we want each refresh to trigger the live pipeline.

## 9. Why Store Before Serving?

The backend could fetch the external APIs and immediately return the result.

But storing in DuckDB is useful:

- We keep a history of snapshots.
- We can audit when refreshes succeeded or failed.
- We can query the latest data quickly.
- We can later build trends from stored snapshots.

This is the bridge from one-off scripts to a real data application.

## 10. Run the Simple Dashboard

Run from the `day_1/04_serving_dashboard/optional_live_serving` folder:

```powershell
..\..\..\.venv\Scripts\python.exe -m uvicorn simple_live_serving_app:app --reload --host 127.0.0.1 --port 8010
```

Then open:

```text
http://127.0.0.1:8010
```

The page refreshes automatically every 2 minutes.

## 11. What You Should Notice

Watch the status line during refresh:

```text
Refreshing: API call, DuckDB insert, redraw PNG
```

That one browser refresh triggers several backend data engineering steps.

This is why a dashboard is more than a static chart.

## 12. Reading the Code Without Panic

When looking at the files, do not read every line first.

Read by responsibility:

| Question | File | Function or Section |
|---|---|---|
| How does the page refresh every 2 minutes? | `simple_live_dashboard.html` | `setInterval(refreshData, REFRESH_MS)` |
| How does the browser call Python? | `simple_live_dashboard.html` | `fetch('/api/refresh')` |
| Which Python function handles that request? | `simple_live_serving_app.py` | `refresh()` |
| Where do we download external APIs? | `simple_live_serving_app.py` | `collect_once()` and `fetch_json(...)` |
| Where do taxi points become area counts? | `simple_live_serving_app.py` | `extract_taxi_counts(...)` |
| Where does rainfall become area rainfall? | `simple_live_serving_app.py` | `extract_rainfall_by_area(...)` |
| Where do we write to DuckDB? | `simple_live_serving_app.py` | `save_latest_to_database(...)` |
| Where is the PNG image drawn? | `simple_live_serving_app.py` | `draw_latest_map(...)` |

This is how larger applications stay understandable: each function has a job.

## 13. Review Questions

**Q1. Why is one API call not enough for a live dashboard?**

Because live data changes. The dashboard needs repeated refreshes.

**Q2. Why write to DuckDB before returning data?**

It preserves snapshots, supports auditability, and makes later queries easier.

**Q3. What is data serving?**

Data serving is making prepared data available to users or applications through APIs, dashboards, or database queries.

**Q4. What is the difference between frontend and backend?**

The frontend runs in the browser and shows the user interface. The backend runs on the server and handles data work, database queries, and API responses.

**Q5. What does JavaScript `fetch()` do?**

`fetch()` sends an HTTP request from the browser to an API endpoint and receives a response.

**Q6. What does `/map.png` return?**

It returns the latest PNG image drawn by the backend.